# Musanga V3 — Notebook 01: Reference data and spatial cross-validation

Goal: create the best full-ROI UAV Musanga probability maps for satellite fractional-cover mapping. This notebook prepares reference geometry, uncertainty policy, spatial folds and candidate windows; it does not train a model or rasterize multi-GB orthomosaics. Four forest groups: Baego, MOS (three mosaics, one group), Yangambi and Yoko. Primary validation: within-site spatial held-out exhaustive circles; final production refits on all annotations.


## 01 — Drive environment and audited input registry


In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib
from pathlib import Path
import json, math, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from rasterio.windows import Window, from_bounds
from rasterio.features import rasterize
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
DATA=next((p for p in [DRIVE/'Colab Notebooks/Musanga/Final/Data',DRIVE/'Musanga/Final/Data'] if p.exists()),None)
assert DATA is not None,'Final/Data directory not found'
AUDIT=DATA.parent/'Outputs/V3/00_input_audit'
OUT=DATA.parent/'Outputs/V3/01_reference_data';OUT.mkdir(parents=True,exist_ok=True)
manifest_path=AUDIT/'00_input_manifest.json'
assert manifest_path.exists(),'Run Notebook 00 through Block 10 first'
A=json.loads(manifest_path.read_text())
MOSAICS={r['file']:r for r in A['mosaics']}
TREE_FILES={site:Path(p) for site,p in A['shared_annotations'].items()}
ROI_FILE=Path(A['roi'])
CIRCLE_FILE=Path(A['exhaustive_annotations']['complete_roi_boundaries'])
EXH_FILE=Path(A['exhaustive_annotations']['positive_polygons'])
SITES=['Baego','MOS','Yangambi','Yoko']
FOLDS=3;PATCH=512;BUFFER_M=15.;SEED=20261015
for p in [ROI_FILE,CIRCLE_FILE,EXH_FILE,*TREE_FILES.values(),*[Path(r['path']) for r in MOSAICS.values()]]:
    assert p.exists(),p
print('Found',len(MOSAICS),'RGB mosaics and',len(TREE_FILES),'shared tree annotation files')
print('Read-only input:',DATA,'\nWrites:',OUT)


## 02 — Repair geometry individually, with traceable quarantines

Invalid geometries repaired in memory if relative area change <= 5%; otherwise excluded individually, never all polygons from the same file. Review quarantines before relying on the final outputs.


In [ ]:
REPAIRS=[]
def parts(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in parts(q)]
    return []
def load_polygons(path):
    raw=gpd.read_file(path)
    assert raw.crs is not None,path
    rows=[];geoms=[]
    for idx,g in enumerate(raw.geometry):
        status='OK';delta=0.
        if g is None or g.is_empty:status='EMPTY'
        elif not g.is_valid or g.geom_type not in ('Polygon','MultiPolygon'):
            try:
                fixed=unary_union(parts(make_valid(g)))
                delta=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if not fixed.is_valid or fixed.is_empty or delta>.05:status='QUARANTINE'
                else:g=fixed;status='REPAIRED'
            except Exception:status='QUARANTINE'
        REPAIRS.append(dict(file=path.name,row=idx,status=status,area_change=delta))
        if status in ('OK','REPAIRED'):rows.append(idx);geoms.append(g)
    output=raw.iloc[rows].copy().reset_index(drop=True)
    output.geometry=geoms
    output['_row_original']=rows
    print(path.name,':',len(output),'kept,',len(raw)-len(output),'quarantined')
    return output
ROI=load_polygons(ROI_FILE);CIRCLES=load_polygons(CIRCLE_FILE)
EXH=load_polygons(EXH_FILE)
TREES={site:load_polygons(path) for site,path in TREE_FILES.items()}
pd.DataFrame(REPAIRS).to_csv(OUT/'01_geometry_repairs.csv',index=False)
assert ROI.geometry.is_valid.all()
for site,g in TREES.items():
    assert 'Class_ID' in g.columns,site
    cls=pd.to_numeric(g.Class_ID,errors='coerce')
    g['_label']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1)).astype('int8')
    print(site,':',int((g._label==1).sum()),'Musanga /',int((g._label==0).sum()),'known negatives')
# As in Notebook 00: candidate circles with no Musanga are not assumed exhaustive.
exh_union=unary_union(EXH.to_crs(CIRCLES.crs).geometry.to_list())
verified=[i for i,g in enumerate(CIRCLES.geometry) if g.intersection(exh_union).area>1]
CIRCLES=CIRCLES.iloc[verified].copy().reset_index(drop=True)
CIRCLES['circle_id']=CIRCLES['_row_original'].astype(int)
assert len(CIRCLES)==22,f'Expected 22 verified circles; got {len(CIRCLES)}'
print('Verified exhaustive circles:',list(CIRCLES.circle_id))


## 03 — Six mosaics nested within four forest groups; hard general ROI


In [ ]:
REG=[];GEO={}
for name,info in MOSAICS.items():
    with rasterio.open(info['path']) as ds:
        assert ds.crs and ds.crs.is_projected and ds.count>=3,name
        footprint=box(*ds.bounds)
        roi=ROI.to_crs(ds.crs)
        eligible=unary_union(roi.geometry.to_list()).intersection(footprint)
        REG.append(dict(mosaic=name,site=info['site'],roi_overlap_m2=float(eligible.area),
                        width=ds.width,height=ds.height,pixel_m=abs(ds.res[0]),crs=str(ds.crs)))
        GEO[name]=dict(transform=ds.transform,crs=ds.crs,width=ds.width,height=ds.height,
                       footprint=footprint,roi=roi,site=info['site'])
REG=pd.DataFrame(REG);display(REG)
REG.to_csv(OUT/'01_mosaic_registry.csv',index=False)
assert (REG.roi_overlap_m2>0).all(),'At least one mosaic lacks ROI coverage: resolve ROI.shp before continuing'


## 04 — Circle-to-mosaic ownership and within-site spatial CV

Three folds per forest group. Spatially nearby circles are preferentially kept in the same fold (maximin spatial seeds, nearest-seed allocation). The three MOS mosaics are **one** forest group, not separate LOSO test sites. The splits are fixed before model fitting.


In [ ]:
LINK=[]
for _,row in CIRCLES.iterrows():
    scores=[]
    for name,d in GEO.items():
        g=gpd.GeoSeries([row.geometry],crs=CIRCLES.crs).to_crs(d['crs']).iloc[0]
        valid=unary_union(d['roi'].geometry.to_list()).intersection(d['footprint'])
        scores.append((float(g.intersection(valid).area),name))
    best_area,name=max(scores)
    assert best_area>0, f'Circle {row.circle_id} has zero general ROI + image intersection'
    frac=best_area/max(row.geometry.area,1e-12)
    if frac<.95:warnings.warn(f'Circle {row.circle_id} valid area only {frac:.1%}')
    LINK.append(dict(circle_id=int(row.circle_id),site=GEO[name]['site'],mosaic=name,
                     roi_coverage_fraction=frac,
                     cover_pct=100*row.geometry.intersection(exh_union).area/row.geometry.area))
LINK=pd.DataFrame(LINK)
assert LINK.groupby('site').size().to_dict()=={'Baego':3,'MOS':7,'Yangambi':7,'Yoko':5}
fold_rows=[]
for site,rows in LINK.groupby('site'):
    ids=rows.circle_id.to_list()
    centers=np.array([[CIRCLES.loc[CIRCLES.circle_id==j].geometry.iloc[0].centroid.x,
                       CIRCLES.loc[CIRCLES.circle_id==j].geometry.iloc[0].centroid.y] for j in ids])
    assert CIRCLES.crs.is_projected
    seeds=[int(np.argmax(np.linalg.norm(centers-centers.mean(axis=0),axis=1)))]
    while len(seeds)<FOLDS:
        dist=np.linalg.norm(centers[:,None,:]-centers[seeds][None,:,:],axis=2).min(axis=1)
        dist[seeds]=-1
        seeds.append(int(np.argmax(dist)))
    for ix,j in enumerate(ids):
        fold=int(np.argmin(np.linalg.norm(centers[ix]-centers[seeds],axis=1)))
        fold_rows.append(dict(circle_id=j,fold=fold))
LINK=LINK.merge(pd.DataFrame(fold_rows),on='circle_id',validate='one_to_one')
assert LINK.groupby('site').fold.nunique().eq(FOLDS).all()
display(LINK.sort_values(['site','fold','circle_id']).round(3))
LINK.to_csv(OUT/'01_circle_spatial_folds.csv',index=False)


## 05 — Single-source raster label generator (native resolution)

`label_window()` returns target, loss weight, annotation source, uncertainty and general ROI masks for a **small native raster window**. `Class_ID=1` means Musanga; other numeric classes mean known non-Musanga. Unknown background outside circles has zero weight. Explicit contradictions have zero training weight. For evaluation, only exhaustive held-out circle pixels count, and contradictions are retained in the full-reference analysis.


In [ ]:
CACHE={}
for name,d in GEO.items():
    circles=CIRCLES.to_crs(d['crs']).copy()
    circles['assigned_mosaic']=circles.circle_id.map(LINK.set_index('circle_id').mosaic)
    circles['fold']=circles.circle_id.map(LINK.set_index('circle_id').fold)
    d['circles']=circles[circles.assigned_mosaic==name].copy()
    d['positive_exhaustive']=EXH.to_crs(d['crs'])
    d['trees']=TREES[d['site']].to_crs(d['crs'])
    CACHE[name]=d
def label_window(name,win,fold=None,role='production'):
    """role='train', 'val', 'production'. Valid pixels only inside ROI.shp."""
    d=CACHE[name]
    shape=(int(win.height),int(win.width))
    transform=rasterio.windows.transform(win,d['transform'])
    bounds=box(*rasterio.windows.bounds(win,d['transform']))
    def burn(geoms):
        intersecting=[g.intersection(bounds) for g in geoms if g is not None and g.intersects(bounds)]
        if not intersecting:return np.zeros(shape,dtype=bool)
        return rasterize([(g,1) for g in intersecting if not g.is_empty],
            out_shape=shape,transform=transform,fill=0,dtype='uint8').astype(bool)
    roi=burn(d['roi'].geometry)
    circles=d['circles']
    if role in ('train','val'):
        assert fold in range(FOLDS)
        circles=circles[circles.fold.ne(fold) if role=='train' else circles.fold.eq(fold)]
    elif role!='production':raise ValueError(role)
    exhaustive=burn(circles.geometry)&roi
    pos_ex=burn(d['positive_exhaustive'].geometry)&exhaustive
    trees=d['trees'];pos_tree=burn(trees.loc[trees._label==1].geometry)
    neg_tree=burn(trees.loc[trees._label==0].geometry)
    uncertain=(pos_ex&neg_tree)|(exhaustive&~pos_ex&pos_tree)|(~exhaustive&pos_tree&neg_tree)
    source=np.zeros(shape,dtype='uint8')
    source[(pos_tree|neg_tree)&~exhaustive]=1
    source[exhaustive]=2
    target=(pos_ex | (~exhaustive&pos_tree)).astype('uint8')
    if role=='val':
        weight=exhaustive.astype('uint8') # full reference includes disagreement for independent QC
    else:
        weight=(roi&(source>0)&~uncertain).astype('uint8')
        if role=='train':
            # also suppress labels within buffer of held-out circles
            holdout=d['circles'][d['circles'].fold==fold]
            forbidden=burn([g.buffer(BUFFER_M) for g in holdout.geometry])
            weight[forbidden]=0
    weight[~roi]=0;source[~roi]=0;target[~roi]=0
    return dict(target=target,weight=weight,source=source,
                uncertain=(uncertain&roi).astype('uint8'),roi=roi.astype('uint8'))
print('Prepared reference rasterization function for',len(CACHE),'mosaics')


## 06 — Compact patch-window candidates

Systematic windows cover exhaustive circles; partial crown windows provide supplementary training examples away from exhaustive circles. **No full-resolution RGB mosaics are copied or rasterized.** Each window points to the original TIFF. MOS crowns are assigned to one mosaic when image footprints overlap.


In [ ]:
PATCH_ROWS=[]
MAX_PARTIAL_PER_MOSAIC=300
for name,d in CACHE.items():
    W,H=d['width'],d['height']
    assert W>=PATCH and H>=PATCH
    for _,circle in d['circles'].iterrows():
        w=from_bounds(*circle.geometry.bounds,transform=d['transform'])
        r0=max(0,int(w.row_off));c0=max(0,int(w.col_off))
        r1=min(H,int(math.ceil(w.row_off+w.height)));c1=min(W,int(math.ceil(w.col_off+w.width)))
        for rr in range((r0//PATCH)*PATCH,r1,PATCH):
            for cc in range((c0//PATCH)*PATCH,c1,PATCH):
                win=Window(cc,rr,min(PATCH,W-cc),min(PATCH,H-rr))
                if not circle.geometry.intersects(box(*rasterio.windows.bounds(win,d['transform']))):continue
                PATCH_ROWS.append(dict(site=d['site'],mosaic=name,kind='exhaustive',
                    circle_id=int(circle.circle_id),fold=int(circle.fold),
                    row=int(rr),col=int(cc),height=int(win.height),width=int(win.width)))
    # Partial-label patches cannot overlap circles or their exclusion buffers.
    exclusion=unary_union([g.buffer(BUFFER_M) for g in d['circles'].geometry]) if len(d['circles']) else None
    trees=d['trees'][d['trees']._label.isin([0,1])]
    rng=np.random.default_rng(SEED+sum(map(ord,name)))
    seen=set()
    for i in rng.permutation(len(trees)):
        p=trees.geometry.iloc[i].representative_point()
        if d['site']=='MOS':
            owners=sorted(k for k,v in CACHE.items() if v['site']=='MOS' and v['footprint'].covers(p))
            if owners and name!=owners[0]:continue
        rr,cc=rasterio.transform.rowcol(d['transform'],p.x,p.y)
        rr=int(max(0,min(H-PATCH,int(rr)-PATCH//2)))
        cc=int(max(0,min(W-PATCH,int(cc)-PATCH//2)))
        key=(rr//(PATCH//2),cc//(PATCH//2))
        if key in seen:continue
        win=Window(cc,rr,PATCH,PATCH)
        shape=box(*rasterio.windows.bounds(win,d['transform']))
        if exclusion is not None and shape.intersects(exclusion):continue
        if not any(g.intersects(shape) for g in d['roi'].geometry):continue
        seen.add(key)
        PATCH_ROWS.append(dict(site=d['site'],mosaic=name,kind='partial',circle_id=-1,fold=-1,
                               row=rr,col=cc,height=PATCH,width=PATCH))
        if len(seen)>=MAX_PARTIAL_PER_MOSAIC:break
PATCHES=pd.DataFrame(PATCH_ROWS).drop_duplicates()
assert len(PATCHES)>0
display(PATCHES.groupby(['site','kind'],as_index=False).size())
PATCHES.to_csv(OUT/'01_candidate_windows.csv',index=False)
print('Reference windows:',len(PATCHES),'(RGB remains in its original TIFFs)')


## 07 — Fold-specific leak-proof train/validation manifests

Entire RGB training windows crossing a held-out circle plus 15 m buffer are excluded, even if the labels outside the circle were usable. The test region stays independent of partial annotation examples. Held-out circle windows are evaluation windows only; their reference pixels are confined to the specific held-out circle.


In [ ]:
ROWS=[]
for fold in range(FOLDS):
    for _,p in PATCHES.iterrows():
        d=CACHE[p.mosaic]
        win=Window(int(p.col),int(p.row),int(p.width),int(p.height))
        footprint=box(*rasterio.windows.bounds(win,d['transform']))
        holdout=d['circles'][d['circles'].fold==fold]
        near_holdout=any(footprint.intersects(g.buffer(BUFFER_M)) for g in holdout.geometry)
        if p.kind=='exhaustive' and int(p.fold)==fold:split='val'
        elif near_holdout:split='excluded_spatial_guard'
        else:split='train'
        ROWS.append(dict(cv_fold=fold,site=p.site,mosaic=p.mosaic,kind=p.kind,
                         circle_id=int(p.circle_id),row=int(p.row),col=int(p.col),
                         width=int(p.width),height=int(p.height),split=split))
SPLITS=pd.DataFrame(ROWS)
# Every site has held-out exhaustive reference in each fold.
counts=SPLITS[SPLITS.split=='val'].groupby(['cv_fold','site']).circle_id.nunique()
assert len(counts)==FOLDS*len(SITES) and counts.min()>=1,counts
for _,row in SPLITS[SPLITS.split=='train'].iterrows():
    d=CACHE[row.mosaic]
    geom=box(*rasterio.windows.bounds(Window(row.col,row.row,row.width,row.height),d['transform']))
    assert not any(geom.intersects(g.buffer(BUFFER_M)) for g in
                   d['circles'].loc[d['circles'].fold==row.cv_fold].geometry),'Training RGB spatial leakage'
display(SPLITS.groupby(['cv_fold','site','split'],as_index=False).size())
SPLITS.to_csv(OUT/'01_cv_window_manifest.csv',index=False)


## 08 — Quick visual QC of label masks

Check reference label categories over a small window for each forest group. Green means Musanga, grey confirmed non-Musanga, yellow explicit disagreement, dark unknown/unusable. All computed at the native orthomosaic pixel size.


In [ ]:
from matplotlib.colors import ListedColormap
examples=[]
for site in SITES:
    g=PATCHES[(PATCHES.site==site)&(PATCHES.kind=='exhaustive')]
    if len(g):examples.append(g.iloc[len(g)//2])
fig,axs=plt.subplots(len(examples),3,figsize=(12,3.3*len(examples)),constrained_layout=True)
if len(examples)==1:axs=axs[None,:]
for i,p in enumerate(examples):
    win=Window(int(p.col),int(p.row),int(p.width),int(p.height))
    lab=label_window(p.mosaic,win,role='production')
    with rasterio.open(MOSAICS[p.mosaic]['path']) as ds:
        rgb=np.moveaxis(ds.read([1,2,3],window=win,masked=True).filled(0),0,-1)
    if rgb.dtype!=np.uint8:
        scale=max(float(np.nanpercentile(rgb,99)),1.)
        rgb=(255*np.clip(rgb.astype('float32')/scale,0,1)).astype('uint8')
    category=np.zeros_like(lab['target'])
    category[lab['weight']>0]=1
    category[(lab['weight']>0)&(lab['target']==1)]=2
    category[lab['uncertain']>0]=3
    category[lab['roi']==0]=4
    axs[i,0].imshow(rgb);axs[i,0].set_title(f'{p.site}: RGB (circle {p.circle_id})')
    axs[i,1].imshow(category,vmin=0,vmax=4,cmap=ListedColormap(
        ['#333333','#cccccc','#2b8c4b','#efb63e','#000000']))
    axs[i,1].set_title('Unknown / non-Musanga / Musanga / conflict / outside ROI')
    axs[i,2].imshow(lab['weight'],cmap='gray',vmin=0,vmax=1)
    axs[i,2].set_title('Trusted training pixels')
    for ax in axs[i]:ax.axis('off')
fig.savefig(OUT/'01_label_QC.png',dpi=150,bbox_inches='tight')
display(fig);plt.close(fig)


## 09 — Site and fold balance audit

The exhaustive circles are the reference for fraction accuracy. Their observed cover is reported separately by forest and CV fold. The 22 circles are not assumed to be a representative probability sample of each entire forest.


In [ ]:
STAT=LINK[['site','mosaic','circle_id','fold','roi_coverage_fraction','cover_pct']].copy()
display(STAT.groupby('site').agg(circles=('circle_id','size'),
    median_cover_pct=('cover_pct','median'),min_cover_pct=('cover_pct','min'),
    max_cover_pct=('cover_pct','max'),min_roi_coverage=('roi_coverage_fraction','min')).round(3))
display(STAT.groupby(['site','fold']).agg(circles=('circle_id','size'),
    mean_cover_pct=('cover_pct','mean')).round(2))
STAT.to_csv(OUT/'01_reference_balance.csv',index=False)
fig,ax=plt.subplots(figsize=(9,4))
for ix,site in enumerate(SITES):
    vals=STAT.loc[STAT.site==site,'cover_pct'].to_numpy()
    ax.scatter(np.full(len(vals),ix)+np.linspace(-.12,.12,len(vals)),vals)
ax.set_xticks(range(len(SITES)),SITES);ax.set_ylabel('Exhaustive circle Musanga cover (%)')
ax.set_title('Within-site reference range (each point = one circle)')
ax.grid(axis='y',alpha=.2)
fig.savefig(OUT/'01_cover_by_site.png',dpi=150,bbox_inches='tight')
display(fig);plt.close(fig)


## 10 — Data contract and model-development hand-off

Notebook 02 should compare exhaustive-only and exhaustive-plus-partial supervision, using **the same folds**. Select pooled versus site-adapted U-Nets based on out-of-fold predictions, fraction bias and calibration, not only segmentation F1. Then refit on all labels for production probability maps **within ROI.shp only**. Include an independent valid RGB/nodata mask while reading TIFF windows.

The target rasterization function is part of this version-pinned notebook; do not redefine label precedence silently.


In [ ]:
contract=dict(schema='musanga_v3_reference_01',input_manifest=str(manifest_path),
  site_groups=SITES,mosaics=list(MOSAICS),circle_count=len(CIRCLES),
  folds=FOLDS,seed=SEED,patch_px=PATCH,buffer_m=BUFFER_M,
  general_roi=str(ROI_FILE),exhaustive_roi=str(CIRCLE_FILE),
  exhaustive_positive=str(EXH_FILE),partial_crowns={s:str(p) for s,p in TREE_FILES.items()},
  label_policy=dict(
    exhaustive='positive polygons Musanga; remaining confirmed background non-Musanga',
    partial='labelled crowns only; blank unknown',unknown='weight zero',
    disagreement='explicit contradictions ignored during training, retained for full-reference validation',
    production='train with all usable annotations after spatial CV selection; never predict outside general ROI'),
  cv_policy='three within-forest spatial folds, MOS treated as one forest group; 15m RGB window exclusion',
  evaluation=['pixel AUC/AP/Brier','circle mean-probability fraction bias/MAE',
              'thresholded cover bias','spatial F1/IoU','site-macro scores'],
  files=['01_geometry_repairs.csv','01_mosaic_registry.csv','01_circle_spatial_folds.csv',
         '01_candidate_windows.csv','01_cv_window_manifest.csv',
         '01_reference_balance.csv','01_label_QC.png'])
(OUT/'01_data_contract.json').write_text(json.dumps(contract,indent=2))
print('Notebook 01 complete. Outputs:')
for f in sorted(OUT.glob('01_*')):print(' ',f.name)
print('No training, full-mosaic raster copying or inference performed.')
